
**Aim**: To utilise a Haar classifier for detecting smiles in OpenCV.


In [ ]:
import math

import cv2
import numpy as np
from matplotlib import pyplot as plt
from skimage.data import lfw_subset
from skimage.feature import draw_haar_like_feature, haar_like_feature_coord, haar_like_feature
from skimage.transform import integral_image
from sklearn.metrics import accuracy_score
from urllib.request import urlopen

import requests
from google.colab.patches import cv2_imshow

In the code cell below, we will list out the urls for our three image, to form our smile detection inputs. We will also set up our OpenCV Haar classifier using the haar_cascade_frontalface_default and haarcascade_smile pre-trained classifiers, for detecting the smiling faces.

In [ ]:
# Preparing a smiling face dataset with three diverse images
# Image URLs
image_urls = [
    "https://images.unsplash.com/photo-1607748862156-7c548e7e98f4?q=80&w=1887&auto=format&fit=crop&ixlib=rb-4.0.3&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D",
    "https://plus.unsplash.com/premium_photo-1661674432167-49e572e92a56?q=80&w=1771&auto=format&fit=crop&ixlib=rb-4.0.3&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D",
    "https://images.unsplash.com/photo-1607749101678-01b521ae7900?q=80&w=1770&auto=format&fit=crop&ixlib=rb-4.0.3&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D"
]

# Setting up an OpenCV Haar classifier using the haarcascade_frontalface_default & haarcascade_smile pre-trained classifiers, for detecting the smiles.
!wget https://raw.githubusercontent.com/opencv/opencv/4.x/data/haarcascades/haarcascade_frontalface_default.xml
!wget https://raw.githubusercontent.com/opencv/opencv/4.x/data/haarcascades/haarcascade_smile.xml

!pip install requests

# Loading the cascade classifiers
face_cascade = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')
smile_cascade = cv2.CascadeClassifier('haarcascade_smile.xml')



**Scenario 1:**

In the code cell below, we will process each image by downloading and then reading and then detecting the faces in the image, and finally detecting the smiles in each face.

we chose 'haarcascade_frontalface_default' pre-trained classifier for our face detection, and 'haarcascade_smile' classifier for our smile detection.

For the face detection, let us try a scaleFactor of 1.3 and
 a value of 5 for minNeighbors.

For smile detection, let us try a scaleFactor of 1.8 and a value of 20 for minNeighbors.

In [ ]:

# Processing each image
for image_url in image_urls:
  # Downloading the image
  img_data = requests.get(image_url).content
  with open('temp_image.jpg', 'wb') as handler:
    handler.write(img_data)

  # Reading the downloaded image
  img = cv2.imread('temp_image.jpg')
  if img is None:
    print(f"Could not read image from: {image_url}")
    continue

  # Converting the image to grayscale
  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

  # Detecting faces in the image
  faces = face_cascade.detectMultiScale(gray, 1.3, 5)
  for (x, y, w, h) in faces:
      # Defining the region of interest for each face
      roi_gray = gray[y:y+h, x:x+w]
      smiles = smile_cascade.detectMultiScale(roi_gray, 1.8, 20)
      # Drawing rectangles around detected smiles
      for (sx, sy, sw, sh) in smiles:
          cv2.rectangle(img, (x+sx, y+sy), (x+sx+sw, y+sy+sh), (255, 0, 0), 2)

  # Displaying the image using cv2_imshow
  cv2_imshow(img)
  cv2.waitKey(0)
  # Closing all OpenCV windows
  cv2.destroyAllWindows()

From the above results, we can see that there are no wrongly placed bounding boxes in any of the three input images but, there are some false negatives (missed smiles) in all the three input images.

**Scenario 2:**

For optimization of our model, and experimenting with multiple combinations of scaleFactor & minNeighbors for both face and smile detection, the below values gave a little bit better results when compared with our Scenario 1.

For the face detection, let us try a scaleFactor of 1.3 and a value of 6 for minNeighbors.

For smile detection, let us try a scaleFactor of 1.18 and a value of 15 for minNeighbors.

In [ ]:

# Experimenting with some other parameters (scaleFactor & minNeighbors) to improve accuracy.

# Processing each image
for image_url in image_urls:
  # Download the image
  img_data = requests.get(image_url).content
  with open('temp_image.jpg', 'wb') as handler:
    handler.write(img_data)

  # Reading the downloaded image
  img = cv2.imread('temp_image.jpg')
  if img is None:
    print(f"Could not read image from: {image_url}")
    continue

  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

  # Detecting faces in the image
  faces = face_cascade.detectMultiScale(gray, 1.3, 6)
  for (x, y, w, h) in faces:
      # Defining the region of interest for each face
      roi_gray = gray[y:y+h, x:x+w]
      smiles = smile_cascade.detectMultiScale(roi_gray, scaleFactor=1.18, minNeighbors=15)
      for (sx, sy, sw, sh) in smiles:
          cv2.rectangle(img, (x+sx, y+sy), (x+sx+sw, y+sy+sh), (255, 0, 0), 2)

  # Displaying the image using cv2_imshow
  cv2_imshow(img)
  cv2.waitKey(0)
  cv2.destroyAllWindows()

From the above results, we can see that we do have more true positives (detected smiles), and also some false positives (detections with no smiles).

Also, we can see that all the smiles were detected in our second input image. The reason is because we are using the haarcascade_frontalface_default classifier for face recognition and the image consists of frontal faces, with good lighting conditions.

**Scenario 3:**

Let us add one more parameter - minSize to both the face detection and smile detection classifiers. The minSize parameter defines the minimum size of the object to be detected. Objects smaller than the value set for minSize are ignored.


In [ ]:
# Experimenting with some other parameters (scaleFactor, minNeighbors & minSize) to improve accuracy.

# Processing each image
for image_url in image_urls:
  # Download the image
  img_data = requests.get(image_url).content
  with open('temp_image.jpg', 'wb') as handler:
    handler.write(img_data)

  # Reading the downloaded image
  img = cv2.imread('temp_image.jpg')
  if img is None:
    print(f"Could not read image from: {image_url}")
    continue

  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

  # Detecting faces in the image
  faces = face_cascade.detectMultiScale(gray, 1.02, 17, minSize=(26,30))
  for (x, y, w, h) in faces:
      # Defining the region of interest for each face
      roi_gray = gray[y:y+h, x:x+w]
      smiles = smile_cascade.detectMultiScale(roi_gray, scaleFactor=1.17, minNeighbors=17, minSize=(23,40))
      for (sx, sy, sw, sh) in smiles:
          cv2.rectangle(img, (x+sx, y+sy), (x+sx+sw, y+sy+sh), (255, 0, 0), 2)

  # Displaying the image using cv2_imshow
  cv2_imshow(img)
  cv2.waitKey(0)
  cv2.destroyAllWindows()